<a href="https://colab.research.google.com/github/bru379/ML/blob/main/CO5420/air_quality_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment setup

In [ ]:
import pandas as pd
import numpy as np
train_raw=pd.read_csv("train_raw.csv")



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315648 entries, 0 to 315647
Data columns (total 18 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   No       315648 non-null  int64  
 1   year     315648 non-null  int64  
 2   month    315648 non-null  int64  
 3   day      315648 non-null  int64  
 4   hour     315648 non-null  int64  
 5   PM2.5    309276 non-null  float64
 6   PM10     311033 non-null  float64
 7   SO2      308661 non-null  float64
 8   NO2      305989 non-null  float64
 9   CO       297542 non-null  float64
 10  O3       305471 non-null  float64
 11  TEMP     315469 non-null  float64
 12  PRES     315468 non-null  float64
 13  DEWP     315464 non-null  float64
 14  RAIN     315472 non-null  float64
 15  wd       314973 non-null  object 
 16  WSPM     315470 non-null  float64
 17  station  315648 non-null  object 
dtypes: float64(11), int64(5), object(2)
memory usage: 43.3+ MB


# Phase 1

* Verified your data infrastructure and folder setup.
* Audited the structural shapes of the file.
* Discovered that every station has the exact same row count ($26,304$ rows).

* Documented where the missing values live across all 12 stations.
 * Established a time-series safe validation plan.



# Structured view of log

In [ ]:
#dataset dimensions

print(f"Total columns found: {train_raw.shape[1]}")
print(f"Total rows found:    {train_raw.shape[0]:,}")

Total columns found: 18
Total rows found:    315,648


### Structural Overview

 information log showing us the data types of each columnand how much memory the dataset takes.


In [ ]:
# Check data types
train_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315648 entries, 0 to 315647
Data columns (total 18 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   No       315648 non-null  int64  
 1   year     315648 non-null  int64  
 2   month    315648 non-null  int64  
 3   day      315648 non-null  int64  
 4   hour     315648 non-null  int64  
 5   PM2.5    309276 non-null  float64
 6   PM10     311033 non-null  float64
 7   SO2      308661 non-null  float64
 8   NO2      305989 non-null  float64
 9   CO       297542 non-null  float64
 10  O3       305471 non-null  float64
 11  TEMP     315469 non-null  float64
 12  PRES     315468 non-null  float64
 13  DEWP     315464 non-null  float64
 14  RAIN     315472 non-null  float64
 15  wd       314973 non-null  object 
 16  WSPM     315470 non-null  float64
 17  station  315648 non-null  object 
dtypes: float64(11), int64(5), object(2)
memory usage: 43.3+ MB


### Missing Values


---


checks every single column and calculates exactly how many blank records exist, sorted from worst to best

In [ ]:
missing_counts=train_raw.isnull().sum()


# Display only the columns that have missing gaps
missing_counts[missing_counts>0].sort_values(ascending=False)

,0
CO,18106
O3,10177
NO2,9659
SO2,6987
PM2.5,6372
PM10,4615
wd,675
DEWP,184
PRES,180
TEMP,179


### Monitoring **Station**

---

distribution

THe log states that data is gathered from 12 distint station ,
Let's make sure all 12 stations are evenly represented with the exact same amount of rows.

In [ ]:
# Verify how many records are logged per monitoring site

train_raw['station'].value_counts()

,count
station,
Aotizhongxin,26304
Changping,26304
Dingling,26304
Dongsi,26304
Guanyuan,26304
Gucheng,26304
Huairou,26304
Nongzhanguan,26304
Shunyi,26304


### Visualizing a Sample Snapshot
look at the actual values inside the first 5 lines of the table


In [ ]:
# Display the first 5 rows of the DataFrame

train_raw.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


### Analyzing Missing Data by Station

In [ ]:
# Calculate the percentage of missing values for each columnon
missing_pct_by_station = (
    train_raw.isnull().groupby(train_raw['station']).mean() * 100
)

# Isolate the main pollutant and weather columns to make the table clean and easy to read
target_columns = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']
station_missing_profile = missing_pct_by_station[target_columns]

# Display the styled table directly in Jupyter
station_missing_profile.round(2)

,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,WSPM
station,,,,,,,,,,,
Aotizhongxin,2.84,2.30,3.02,3.09,6.18,5.54,0.01,0.01,0.01,0.01,0.01
Changping,2.51,1.84,1.86,2.07,5.22,1.87,0.13,0.13,0.13,0.13,0.12
Dingling,1.89,1.86,2.02,4.00,6.71,2.87,0.13,0.13,0.13,0.13,0.12
Dongsi,1.41,0.91,1.32,4.83,10.83,1.24,0.01,0.01,0.01,0.01,0.01
Guanyuan,1.62,1.10,1.43,2.13,5.83,4.04,0.01,0.01,0.01,0.01,0.01
Gucheng,1.87,1.06,1.57,2.01,4.85,2.41,0.13,0.13,0.13,0.10,0.12
Huairou,3.10,2.46,3.19,4.92,4.58,3.72,0.13,0.13,0.13,0.14,0.14
Nongzhanguan,1.85,1.25,1.28,1.97,3.68,1.48,0.01,0.01,0.01,0.01,0.01
Shunyi,2.23,1.05,3.31,3.53,6.67,2.19,0.13,0.13,0.14,0.13,0.13


# Cleaning the gaps

---
imputation process



*   groupby 'station' we used this to sepearte the log data of diffrent station to ensure that data of one station steal from next  station first hour


*   forwardfill: when it sees the NaN copies the values from the row directly above from it

*   backwardfill:when it sees the NaN copies the values from the row directly below from it


*   forwardfill strips the text column to focus on numbers so we have to again attach the station column again it copy and paste from our sorted copy





In [ ]:
train_clean=train_raw.copy()

#Sorting the Timeline
train_clean = train_clean.sort_values(by=['station', 'year', 'month', 'day', 'hour']).reset_index(drop=True)

#  Apply forward fill (carry last hour forward), then backward fill (catch anything missing at row 0)
# We group by station so data from one station never accidentally bleeds into another
#  Apply Forward Fill per station (We use lambda to keep the dataframe structure intact)
columns_to_fill = [c for c in train_clean.columns if c != 'station']
train_clean[columns_to_fill] = train_clean.groupby('station', group_keys=False)[columns_to_fill].ffill()
train_clean[columns_to_fill] = train_clean.groupby('station', group_keys=False)[columns_to_fill].bfill()


#re     attach the text column station
train_clean['station'] = train_raw.sort_values(by=['station', 'year', 'month', 'day', 'hour'])['station'].values


In [ ]:
print("Remaining missing values:", train_clean.isnull().sum().sum())

Remaining missing values: 0
